# TASK 0 — Setup & Sanity Checks for Indicator Analyses

**Goal**: Prepare shared data structures and verify assumptions before running any indicator analysis.

## Steps
1. Load frozen DreaMS SSL embeddings (1024-d per spectrum)
2. Load all RDKit descriptors (201 continuous descriptors)
3. Load molecule-level splits (validation = probing_test)
4. Merge descriptors with validation spectra on inchikey
5. Assert data quality:
   - No descriptor has NaNs or infinite values
   - Every spectrum maps to exactly one molecule (via InChIKey)
   - No molecule appears in multiple splits
6. Select **validation split** (probing_test) for all indicator analyses
7. Compute and cache:
   - spectrum → molecule mapping
   - molecule → list of spectrum indices
8. Print summary to `results/indicators/setup_summary.txt`

## Data Sources
- SSL embeddings: `MassSpecGym_splits/probing_test.parquet` (ssl_embedding column)
- Descriptors: `massspecgym_complete/all_rdkit_descriptors.parquet` (201 RDKit descriptors)
- Validation set: `probing_test.parquet` (45,185 spectra)

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

✓ Imports complete


## 1. Load Data

In [9]:
# Paths
data_dir = Path('../data/processed')
splits_dir = data_dir / 'MassSpecGym_splits'
complete_dir = data_dir / 'massspecgym_complete'
results_dir = Path('../results/indicators')
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {data_dir}")

print(f"Splits directory: {splits_dir}")
print(f"Results directory: {results_dir}")

Data directory: ../data/processed
Splits directory: ../data/processed/MassSpecGym_splits
Results directory: ../results/indicators


In [10]:
# Load validation split (probing_test) with embeddings and descriptors
print("Loading validation split (probing_test.parquet)...")
df_val = pd.read_parquet(splits_dir / 'probing_test.parquet')

print(f"  Shape: {df_val.shape}")
print(f"  Columns: {len(df_val.columns)}")
print(f"  Has ssl_embedding: {'ssl_embedding' in df_val.columns}")
print(f"  Has inchikey: {'inchikey' in df_val.columns}")
print(f"\n  Sample columns: {list(df_val.columns[:15])}...")

Loading validation split (probing_test.parquet)...
  Shape: (45185, 31)
  Columns: 31
  Has ssl_embedding: True
  Has inchikey: True

  Sample columns: ['identifier', 'mzs', 'intensities', 'smiles', 'inchikey', 'formula', 'precursor_formula', 'parent_mass', 'precursor_mz', 'adduct', 'instrument_type', 'collision_energy', 'simulation_challenge', 'scaffold_id', 'murcko_scaffold']...


In [11]:
# Load all RDKit descriptors
print("\nLoading RDKit descriptors (all_rdkit_descriptors.parquet)...")
df_descriptors = pd.read_parquet(complete_dir / 'all_rdkit_descriptors.parquet')

print(f"  Shape: {df_descriptors.shape}")
print(f"  Columns: {len(df_descriptors.columns)}")
print(f"  Has inchikey: {'inchikey' in df_descriptors.columns}")


Loading RDKit descriptors (all_rdkit_descriptors.parquet)...
  Shape: (31602, 209)
  Columns: 209
  Has inchikey: False


In [12]:
# Identify descriptor columns from all_rdkit_descriptors
# (Exclude identifier columns)
print("Identifying RDKit descriptors...")

exclude_cols = {'smiles'}
descriptor_cols = [col for col in df_descriptors.columns if col not in exclude_cols]

print(f"  Total descriptor columns: {len(descriptor_cols)}")
print(f"  First 10 descriptors: {sorted(descriptor_cols)[:10]}")
print(f"  Last 10 descriptors: {sorted(descriptor_cols)[-10:]}")

Identifying RDKit descriptors...
  Total descriptor columns: 208
  First 10 descriptors: ['BCUT2D_CHGHI', 'BCUT2D_CHGLO', 'BCUT2D_LOGPHI', 'BCUT2D_LOGPLOW', 'BCUT2D_MRHI', 'BCUT2D_MRLOW', 'BCUT2D_MWHI', 'BCUT2D_MWLOW', 'BalabanJ', 'BertzCT']
  Last 10 descriptors: ['fr_sulfonamd', 'fr_sulfone', 'fr_term_acetylene', 'fr_tetrazole', 'fr_thiazole', 'fr_thiocyan', 'fr_thiophene', 'fr_unbrch_alkane', 'fr_urea', 'qed']


## 2. Extract SSL Embeddings

In [13]:
# Extract embeddings as numpy array
print("Extracting SSL embeddings...")
ssl_embeddings = np.vstack(df_val['ssl_embedding'].values)

print(f"  Embeddings shape: {ssl_embeddings.shape}")
print(f"  Embedding dimension: {ssl_embeddings.shape[1]}")
print(f"  Number of spectra: {ssl_embeddings.shape[0]}")
print(f"  Dtype: {ssl_embeddings.dtype}")

# Basic stats
print(f"\n  Mean: {ssl_embeddings.mean():.6f}")
print(f"  Std: {ssl_embeddings.std():.6f}")
print(f"  Min: {ssl_embeddings.min():.6f}")
print(f"  Max: {ssl_embeddings.max():.6f}")

Extracting SSL embeddings...
  Embeddings shape: (45185, 1024)
  Embedding dimension: 1024
  Number of spectra: 45185
  Dtype: float32

  Mean: 0.001735
  Std: 1.355451
  Min: -12.678819
  Max: 27.204519


## 3. Merge Descriptors with Validation Data

In [14]:
# Merge descriptors with validation spectra on smiles
print("\nMerging descriptors with validation spectra (on smiles)...")
print(f"  Before merge - Validation spectra: {len(df_val):,}")
print(f"  Before merge - Unique molecules in descriptors: {df_descriptors['smiles'].nunique():,}")

# Check overlap
overlap = set(df_val['smiles']) & set(df_descriptors['smiles'])
print(f"  SMILES overlap: {len(overlap):,} / {df_val['smiles'].nunique():,} validation molecules")

df_merged = df_val.merge(df_descriptors, on='smiles', how='left', suffixes=('', '_desc'))

print(f"  After merge - Shape: {df_merged.shape}")
print(f"  Rows with missing descriptors: {df_merged[descriptor_cols[0]].isna().sum():,}")


Merging descriptors with validation spectra (on smiles)...
  Before merge - Validation spectra: 45,185
  Before merge - Unique molecules in descriptors: 31,602
  SMILES overlap: 6,147 / 6,147 validation molecules
  After merge - Shape: (45185, 239)
  Rows with missing descriptors: 0


## 4. Data Quality Assertions

In [15]:
print("="*80)
print("RUNNING DATA QUALITY CHECKS")
print("="*80)

errors = []
warnings_list = []

# Check 1: No NaNs in descriptors
print("\n[1/5] Checking for NaN values in descriptors...")
nan_counts = df_merged[descriptor_cols].isna().sum()
descriptors_with_nans = nan_counts[nan_counts > 0]

if len(descriptors_with_nans) > 0:
    errors.append(f"Found {len(descriptors_with_nans)} descriptors with NaN values")
    print(f"  ❌ FAILED: {len(descriptors_with_nans)} descriptors have NaN values")
    print(f"     Top 10 descriptors with NaNs:")
    for desc, count in descriptors_with_nans.head(10).items():
        print(f"       {desc}: {count} NaNs ({count/len(df_merged)*100:.2f}%)")
else:
    print("  ✓ PASSED: No NaN values in descriptors")

# Check 2: No infinite values
print("\n[2/5] Checking for infinite values in descriptors...")
inf_counts = np.isinf(df_merged[descriptor_cols].values).sum(axis=0)
descriptors_with_infs = pd.Series(inf_counts, index=descriptor_cols)
descriptors_with_infs = descriptors_with_infs[descriptors_with_infs > 0]

if len(descriptors_with_infs) > 0:
    errors.append(f"Found {len(descriptors_with_infs)} descriptors with infinite values")
    print(f"  ❌ FAILED: {len(descriptors_with_infs)} descriptors have infinite values")
    for desc, count in descriptors_with_infs.head(10).items():
        print(f"       {desc}: {count} infs ({count/len(df_merged)*100:.2f}%)")
else:
    print("  ✓ PASSED: No infinite values in descriptors")

# Check 3: Every spectrum maps to exactly one molecule
print("\n[3/5] Checking spectrum → molecule mapping...")
null_inchikeys = df_merged['inchikey'].isna().sum()
if null_inchikeys > 0:
    errors.append(f"Found {null_inchikeys} spectra without InChIKey")
    print(f"  ❌ FAILED: {null_inchikeys} spectra have no InChIKey")
else:
    print(f"  ✓ PASSED: All {len(df_merged)} spectra have InChIKey")

# Check 4: No embedding has NaNs
print("\n[4/5] Checking for NaN values in SSL embeddings...")
nan_embeddings = np.isnan(ssl_embeddings).any(axis=1).sum()
if nan_embeddings > 0:
    errors.append(f"Found {nan_embeddings} embeddings with NaN values")
    print(f"  ❌ FAILED: {nan_embeddings} embeddings have NaN values")
else:
    print("  ✓ PASSED: No NaN values in embeddings")

# Check 5: Verify this is the validation split only
print("\n[5/5] Verifying split identity...")
if 'fold' in df_val.columns:
    fold_counts = df_val['fold'].value_counts()
    print(f"  Fold distribution: {fold_counts.to_dict()}")
    if len(fold_counts) > 1:
        warnings_list.append(f"Multiple folds found in validation data: {fold_counts.to_dict()}")
        print(f"  ⚠️  WARNING: Multiple folds present")
    else:
        print(f"  ✓ PASSED: Single fold only ({fold_counts.index[0]})")
else:
    print("  ℹ️  INFO: No 'fold' column in data")

print("\n" + "="*80)
if errors:
    print("❌ DATA QUALITY CHECK FAILED")
    print(f"   {len(errors)} error(s) found:")
    for err in errors:
        print(f"   - {err}")
else:
    print("✅ ALL DATA QUALITY CHECKS PASSED")

if warnings_list:
    print(f"\n⚠️  {len(warnings_list)} warning(s):")
    for warn in warnings_list:
        print(f"   - {warn}")

print("="*80)

RUNNING DATA QUALITY CHECKS

[1/5] Checking for NaN values in descriptors...
  ✓ PASSED: No NaN values in descriptors

[2/5] Checking for infinite values in descriptors...
  ✓ PASSED: No infinite values in descriptors

[3/5] Checking spectrum → molecule mapping...
  ✓ PASSED: All 45185 spectra have InChIKey

[4/5] Checking for NaN values in SSL embeddings...
  ✓ PASSED: No NaN values in embeddings

[5/5] Verifying split identity...
  Fold distribution: {'test': 45185}
  ✓ PASSED: Single fold only (test)

✅ ALL DATA QUALITY CHECKS PASSED


## 5. Compute Spectrum ↔ Molecule Mappings

In [16]:
print("\nComputing spectrum ↔ molecule mappings...")

# Spectrum index → InChIKey
spectrum_to_molecule = df_merged['inchikey'].values
print(f"  ✓ spectrum_to_molecule: {len(spectrum_to_molecule)} entries")

# InChIKey → list of spectrum indices
molecule_to_spectra = defaultdict(list)
for idx, inchikey in enumerate(spectrum_to_molecule):
    molecule_to_spectra[inchikey].append(idx)

molecule_to_spectra = dict(molecule_to_spectra)  # Convert to regular dict
print(f"  ✓ molecule_to_spectra: {len(molecule_to_spectra)} unique molecules")

# Statistics
spectra_per_molecule = [len(v) for v in molecule_to_spectra.values()]
print(f"\n  Spectra per molecule:")
print(f"    Mean: {np.mean(spectra_per_molecule):.2f}")
print(f"    Median: {np.median(spectra_per_molecule):.0f}")
print(f"    Min: {np.min(spectra_per_molecule)}")
print(f"    Max: {np.max(spectra_per_molecule)}")
print(f"    Molecules with 1 spectrum: {sum(1 for x in spectra_per_molecule if x == 1):,}")
print(f"    Molecules with >1 spectrum: {sum(1 for x in spectra_per_molecule if x > 1):,}")


Computing spectrum ↔ molecule mappings...
  ✓ spectrum_to_molecule: 45185 entries
  ✓ molecule_to_spectra: 5706 unique molecules

  Spectra per molecule:
    Mean: 7.92
    Median: 3
    Min: 1
    Max: 252
    Molecules with 1 spectrum: 852
    Molecules with >1 spectrum: 4,854


## 6. Create Clean Dataset for Indicators

In [17]:
print("\nCreating clean dataset for indicator analyses...")

# Store key data
indicator_data = {
    'embeddings': ssl_embeddings,  # (n_spectra, 1024)
    'descriptors': df_merged[descriptor_cols].values,  # (n_spectra, n_descriptors)
    'descriptor_names': descriptor_cols,
    'spectrum_to_molecule': spectrum_to_molecule,  # array of InChIKeys
    'molecule_to_spectra': molecule_to_spectra,  # dict: InChIKey → list of indices
    'inchikeys': df_merged['inchikey'].values,
    'smiles': df_merged['smiles'].values if 'smiles' in df_merged.columns else None,
    'n_spectra': len(df_merged),
    'n_molecules': len(molecule_to_spectra),
    'n_descriptors': len(descriptor_cols),
    'embedding_dim': ssl_embeddings.shape[1]
}

print(f"  ✓ Created indicator_data dictionary")
print(f"    Keys: {list(indicator_data.keys())}")
print(f"\n  Summary:")
print(f"    Spectra: {indicator_data['n_spectra']:,}")
print(f"    Molecules: {indicator_data['n_molecules']:,}")
print(f"    Descriptors: {indicator_data['n_descriptors']}")
print(f"    Embedding dimension: {indicator_data['embedding_dim']}")


Creating clean dataset for indicator analyses...
  ✓ Created indicator_data dictionary
    Keys: ['embeddings', 'descriptors', 'descriptor_names', 'spectrum_to_molecule', 'molecule_to_spectra', 'inchikeys', 'smiles', 'n_spectra', 'n_molecules', 'n_descriptors', 'embedding_dim']

  Summary:
    Spectra: 45,185
    Molecules: 5,706
    Descriptors: 208
    Embedding dimension: 1024


## 7. Cache & Serialize Indicator Data

In [21]:
import joblib

# Save indicator_data for downstream notebooks
cache_path = results_dir / 'indicator_data.pkl'
joblib.dump(indicator_data, cache_path)

print(f"\n✅ Cached indicator_data to disk:")
print(f"   Path: {cache_path}")
print(f"   Size: {cache_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\n   Other notebooks can load it with:")
print(f"   ```python")
print(f"   import joblib")
print(f"   indicator_data = joblib.load('results/indicators/indicator_data.pkl')")
print(f"   ```")


✅ Cached indicator_data to disk:
   Path: ../results/indicators/indicator_data.pkl
   Size: 249.48 MB

   Other notebooks can load it with:
   ```python
   import joblib
   indicator_data = joblib.load('results/indicators/indicator_data.pkl')
   ```


## 7. Serialize & Cache Indicator Data

In [22]:
import joblib

# Save indicator_data for downstream notebooks
cache_path = results_dir / 'indicator_data.pkl'
joblib.dump(indicator_data, cache_path)

print(f"✅ Saved cached indicator_data to: {cache_path}")
print(f"   File size: {cache_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"\nOther notebooks can load it with:")
print(f'   from pathlib import Path')
print(f'   import joblib')
print(f'   indicator_data = joblib.load(Path("../results/indicators/indicator_data.pkl"))')

✅ Saved cached indicator_data to: ../results/indicators/indicator_data.pkl
   File size: 249.48 MB

Other notebooks can load it with:
   from pathlib import Path
   import joblib
   indicator_data = joblib.load(Path("../results/indicators/indicator_data.pkl"))


## 8. Save Summary Report

In [23]:
# Write summary to file
summary_path = results_dir / 'setup_summary.txt'

with open(summary_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TASK 0 — Setup & Sanity Checks Summary\n")
    f.write("="*80 + "\n\n")
    
    f.write("## Data Sources\n")
    f.write(f"- Validation split: MassSpecGym_splits/probing_test.parquet\n")
    f.write(f"- Descriptors: Already included in probing_test.parquet\n")
    f.write(f"\n")
    
    f.write("## Dataset Statistics\n")
    f.write(f"- Total spectra: {indicator_data['n_spectra']:,}\n")
    f.write(f"- Unique molecules: {indicator_data['n_molecules']:,}\n")
    f.write(f"- RDKit descriptors: {indicator_data['n_descriptors']}\n")
    f.write(f"- Embedding dimension: {indicator_data['embedding_dim']}\n")
    f.write(f"\n")
    
    f.write("## Spectra per Molecule\n")
    f.write(f"- Mean: {np.mean(spectra_per_molecule):.2f}\n")
    f.write(f"- Median: {np.median(spectra_per_molecule):.0f}\n")
    f.write(f"- Range: [{np.min(spectra_per_molecule)}, {np.max(spectra_per_molecule)}]\n")
    f.write(f"- Molecules with 1 spectrum: {sum(1 for x in spectra_per_molecule if x == 1):,}\n")
    f.write(f"- Molecules with >1 spectra: {sum(1 for x in spectra_per_molecule if x > 1):,}\n")
    f.write(f"\n")
    
    f.write("## Data Quality Checks\n")
    if errors:
        f.write(f"❌ FAILED ({len(errors)} error(s)):\n")
        for err in errors:
            f.write(f"  - {err}\n")
    else:
        f.write("✅ ALL CHECKS PASSED\n")
    
    if warnings_list:
        f.write(f"\n⚠️  Warnings ({len(warnings_list)}):")
        for warn in warnings_list:
            f.write(f"  - {warn}\n")
    
    f.write(f"\n")
    f.write("## Descriptor Issues\n")
    if len(descriptors_with_nans) > 0:
        f.write(f"Descriptors with NaN values: {len(descriptors_with_nans)}\n")
        for desc, count in descriptors_with_nans.head(20).items():
            f.write(f"  - {desc}: {count} NaNs ({count/len(df_merged)*100:.2f}%)\n")
    else:
        f.write("No descriptors with NaN values\n")
    
    f.write(f"\n")
    f.write("## Cached Objects\n")
    f.write("Available in `indicator_data` dictionary:\n")
    for key in indicator_data.keys():
        if isinstance(indicator_data[key], np.ndarray):
            f.write(f"  - {key}: {indicator_data[key].shape}\n")
        elif isinstance(indicator_data[key], dict):
            f.write(f"  - {key}: dict with {len(indicator_data[key])} entries\n")
        elif isinstance(indicator_data[key], list):
            f.write(f"  - {key}: list with {len(indicator_data[key])} entries\n")
        else:
            f.write(f"  - {key}: {indicator_data[key]}\n")
    
    f.write(f"\n")
    f.write("="*80 + "\n")
    f.write("TASK 0 COMPLETE — Ready for indicator analyses (IND1, IND2, IND3)\n")
    f.write("="*80 + "\n")

print(f"\n✅ Summary saved to: {summary_path}")
print(f"\n📊 TASK 0 COMPLETE")
print(f"   You can now proceed with indicator analyses:")
print(f"   - IND1: Nearest-neighbor descriptor consistency")
print(f"   - IND2: Clustering purity")
print(f"   - IND3: Structural separation")


✅ Summary saved to: ../results/indicators/setup_summary.txt

📊 TASK 0 COMPLETE
   You can now proceed with indicator analyses:
   - IND1: Nearest-neighbor descriptor consistency
   - IND2: Clustering purity
   - IND3: Structural separation


## 8. Display Summary

In [24]:
## 9. Display Summary